## Part 2

In [ ]:
import pandas as pd

# Read the project summary CSV
df = pd.read_csv('kha5_project_summary.csv', sep=';')

# Convert the Unix timestamp in the "time" column to a date
df['date'] = pd.to_datetime(df['time'], unit='s')

# Get the month for each commit
df['Month'] = df['date'].dt.to_period('M')

# Count commits for each month
monthly = df.groupby('Month').size()

# Create every month from the first commit to the last commit
all_months = pd.period_range(
    start=monthly.index.min(),
    end=monthly.index.max(),
    freq='M'
)

# Add months with zero commits
monthly = monthly.reindex(all_months, fill_value=0)

# Create the final DataFrame with the required headers
monthly_df = monthly.reset_index()
monthly_df.columns = ['Month', '#Commits']

# Save the CSV
monthly_df.to_csv('kha5_commits_timeseries.csv', sep=';', index=False)

print("Created monthly_commits.csv")
print(monthly_df.to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV containing commits from all projects
df = pd.read_csv("kha5_project_summary.csv", sep=";")

# Convert Unix timestamps to dates
df["date"] = pd.to_datetime(df["time"], unit="s")

# Create a separate time series and plot for each project
for project in df["project_wocid"].unique():

    # Select commits belonging to this project
    project_df = df[df["project_wocid"] == project].copy()

    # Group commits by month
    project_df["Month"] = project_df["date"].dt.to_period("M")
    monthly = project_df.groupby("Month").size()

    # Include every month between the first and last commit
    all_months = pd.period_range(
        start=monthly.index.min(),
        end=monthly.index.max(),
        freq="M"
    )
    monthly = monthly.reindex(all_months, fill_value=0)

    # Convert months to YYYY-MM strings
    months = monthly.index.astype(str)

    # Create the plot
    plt.figure(figsize=(10, 5))
    plt.plot(months, monthly.values, marker="o", linestyle="-")

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(project)
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Save a separate PNG for this project
    filename = f"kha5_timeseries_{project}.png"
    plt.savefig(filename, dpi=300)
    plt.close()

    print(f"Saved: {filename}")

In [ ]:
import pandas as pd

# Read the existing commit summary CSV
df = pd.read_csv("kha5_project_summary.csv", sep=";")

# Convert Unix timestamps to dates
df["date"] = pd.to_datetime(df["time"], unit="s")
df["Month"] = df["date"].dt.to_period("M")

# Fill in GitHub information for each project.
github_info = {
    "socr_socrat": {
        "number of stars": 40,
        "number of forks": 55,
        "last commit date": "6/22/25"
    },
    "bbn-q_qgl": {
        "number of stars": 33,
        "number of forks": 12,
        "last commit date": "5/21/26"
    },
    "microsoft_sptag": {
        "number of stars": 5000,
        "number of forks": 621,
        "last commit date": "8/28/26"
    },
    "allenai_scispacy": {
        "number of stars": 2000,
        "number of forks": 260,
        "last commit date": "12/4/25"
    },
    "clawpack_pyclaw": {
        "number of stars": 187,
        "number of forks": 109,
        "last commit date": "6/25/26"
    },
    "isce-framework_isce2": {
        "number of stars": 669,
        "number of forks": 312,
        "last commit date": "6/30/26"
    },
    "facebookresearch_pytorch3d": {
        "number of stars": 10000,
        "number of forks": 1500,
        "last commit date": "9/15/26"
    },
    "mundialis_actinia_core": {
        "number of stars": 96,
        "number of forks": 25,
        "last commit date": "9/17/26"
    },
    "unidata_netcdf-cxx4": {
        "number of stars": 152,
        "number of forks": 57,
        "last commit date": "9/24/26"
    },
    "astro-informatics_ssht": {
        "number of stars": 12,
        "number of forks": 2,
        "last commit date": "4/1/25"
    }
}

activity_patterns = {
    "socr_socrat": "declining",
    "bbn-q_qgl": "irregular",
    "microsoft_sptag": "cyclical",
    "allenai_scispacy": "declining",
    "clawpack_pyclaw": "declining",
    "isce-framework_isce2": "declining",
    "facebookresearch_pytorch3d": "declining",
    "mundialis_actinia_core": "cyclical",
    "unidata_netcdf-cxx4": "irregular",
    "astro-informatics_ssht": "declining"
}

results = []

# Process each project separately
for project in df["project_wocid"].unique():
    p = df[df["project_wocid"] == project].copy()

    # Basic project statistics
    number_of_commits = p["commit_sha1"].nunique()
    number_of_authors = p["author"].nunique()
    max_time = p["time"].max()
    min_time = p["time"].min()

    # Count commits in each month
    monthly = p.groupby("Month").size()

    # Include every month from first to last commit
    all_months = pd.period_range(
        monthly.index.min(),
        monthly.index.max(),
        freq="M"
    )
    monthly = monthly.reindex(all_months, fill_value=0)

    # Find all consecutive zero-commit periods
    gaps = []
    gap_start = None

    for i, count in enumerate(monthly.values):
        if count == 0 and gap_start is None:
            gap_start = i

        # Close a gap when a month has commits
        if count > 0 and gap_start is not None:
            gap_end = i - 1
            gaps.append((gap_start, gap_end))
            gap_start = None

    # Handle a gap that continues through the last month
    if gap_start is not None:
        gaps.append((gap_start, len(monthly) - 1))

    # Find the longest gap; ties use the earliest gap
    if gaps:
        longest_start, longest_end = max(
            gaps,
            key=lambda g: g[1] - g[0] + 1
        )
        longest_length = longest_end - longest_start + 1

        longest_gap_start = str(all_months[longest_start])
        longest_gap_end = str(all_months[longest_end])

        # Count commits in months after the longest gap ends
        commits_after_gap = int(
            monthly.iloc[longest_end + 1:].sum()
        )
    else:
        longest_gap_start = ""
        longest_gap_end = ""
        longest_length = 0
        commits_after_gap = 0

    # Count gaps with at least 3 consecutive zero-commit months
    number_of_gaps = sum(
        1 for start, end in gaps
        if end - start + 1 >= 3
    )

    # GitHub information; blanks if not yet filled in
    gh = github_info.get(project, {})

    activity_pattern = activity_patterns.get(project, "")

    results.append({
        "project": project,
        "number of commits": number_of_commits,
        "number of authors": number_of_authors,
        "max time": max_time,
        "min time": min_time,
        "number of stars": gh.get("number of stars"),
        "number of forks": gh.get("number of forks"),
        "last commit date": gh.get("last commit date", ""),
        "LongestGapStart (YYYY-MM)": longest_gap_start,
        "LongestGapEnd (YYYY-MM)": longest_gap_end,
        "LongestGapLength (months)": longest_length,
        "NumCommitsAfterLastGap": commits_after_gap,
        "NumberOfGapsInTimeline": number_of_gaps,
        "ActivityPattern": activity_pattern
    })

# Create and save the final CSV
stats_df = pd.DataFrame(results)

stats_df.to_csv(
    "kha5_project_stats.csv",
    sep=";",
    index=False
)

print(stats_df.to_string(index=False))
print("\nCreated kha5_project_stats.csv")

## INTERPRETATION

Declining: Activity was higher in the earlier years and tapered off in more recent years. This can indicate either work is complete or no updates are wanted or needed

Irregular: Activity is inconsistent. There's periods of high activity followed by periods of low activity/inactivity but not in a pattern. This can indicate either the creator takes long periods to develop or don't work on it consistently.

Cyclical: There's consistent predictable periods of high activity then low activity. This can indicate that the contributor may work in spurts instead of in a steady pattern